In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


DATA_PATH = "/kaggle/input/datasets/atharvasoundankar/smart-farming-sensor-data-for-yield-prediction/Smart_Farming_Crop_Yield_2024.csv"

RANDOM_STATE = 42

FEATURES = [
    "soil_moisture_%",
    "temperature_C",
    "rainfall_mm",
    "humidity_%",
    "sunlight_hours",
    "NDVI_index"
]


def load_data(path):
    df = pd.read_csv(path)
    return df


def create_target(df):
    df = df.copy()

    df["water_score"] = (
        (100 - df["soil_moisture_%"]) * 0.45 +
        df["temperature_C"] * 0.25 +
        (100 - df["humidity_%"]) * 0.15 +
        (100 - df["rainfall_mm"]) * 0.10 +
        df["sunlight_hours"] * 0.05
    )

    df["needs_water"] = (df["water_score"] > df["water_score"].median()).astype(int)

    return df


def inspect_data(df):
    print("Dataset shape:", df.shape)
    print("\nColumns:")
    print(df.columns)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nPreview:")
    display(df.head())

    print("\nTarget balance:")
    print(df["needs_water"].value_counts(normalize=True))


def plot_class_balance(df):
    counts = df["needs_water"].value_counts().sort_index()

    plt.figure(figsize=(6, 4))
    plt.bar(["Do Not Water", "Water"], counts.values)
    plt.title("Target Class Balance")
    plt.ylabel("Number of Samples")
    plt.show()


def plot_feature_distributions(df, features):
    for feature in features:
        plt.figure(figsize=(6, 4))
        plt.hist(df[feature], bins=30)
        plt.title(f"Distribution of {feature}")
        plt.xlabel(feature)
        plt.ylabel("Frequency")
        plt.show()


def plot_correlation_matrix(df, features):
    corr = df[features + ["needs_water"]].corr()

    plt.figure(figsize=(8, 6))
    plt.imshow(corr)
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Feature Correlation Matrix")
    plt.show()


def build_models():
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
        ]),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=RANDOM_STATE
        ),

        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE
        )
    }

    return models


def evaluate_models(models, X_train, X_test, y_train, y_test):
    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds),
            "Recall": recall_score(y_test, preds),
            "F1 Score": f1_score(y_test, preds)
        })

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)
        print(classification_report(y_test, preds))

        cm = confusion_matrix(y_test, preds)
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Do Not Water", "Water"]
        )
        disp.plot()
        plt.title(f"Confusion Matrix: {name}")
        plt.show()

    results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)
    return results_df


def plot_model_comparison(results_df):
    plt.figure(figsize=(8, 5))
    plt.bar(results_df["Model"], results_df["F1 Score"])
    plt.title("Model Comparison by F1 Score")
    plt.ylabel("F1 Score")
    plt.xticks(rotation=20)
    plt.show()


def get_feature_importance(model, features):
    if isinstance(model, Pipeline):
        final_model = model.named_steps["model"]
        importances = np.abs(final_model.coef_[0])
    else:
        importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": features,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    return importance_df


def plot_feature_importance(importance_df):
    plt.figure(figsize=(8, 5))
    plt.barh(importance_df["Feature"], importance_df["Importance"])
    plt.gca().invert_yaxis()
    plt.title("Feature Importance")
    plt.xlabel("Importance")
    plt.show()


def predict_sample(model, sample):
    prediction = model.predict(sample)[0]

    if hasattr(model, "predict_proba"):
        confidence = model.predict_proba(sample)[0][1]
    else:
        confidence = None

    print("\nSample Farm Input:")
    display(sample)

    if prediction == 1:
        print("AI Decision: Water the plant")
    else:
        print("AI Decision: Do not water")

    if confidence is not None:
        print("Confidence that watering is needed:", round(confidence * 100, 2), "%")


df = load_data(DATA_PATH)
df = create_target(df)

inspect_data(df)

plot_class_balance(df)
plot_feature_distributions(df, FEATURES)
plot_correlation_matrix(df, FEATURES)

X = df[FEATURES]
y = df["needs_water"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

models = build_models()

results_df = evaluate_models(models, X_train, X_test, y_train, y_test)

print("\nFinal Model Comparison:")
display(results_df)

plot_model_comparison(results_df)

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("\nBest Performing Model:", best_model_name)

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring="f1"
)

print("\n5-Fold Cross-Validation F1 Scores:", cv_scores)
print("Average CV F1 Score:", round(cv_scores.mean(), 4))

importance_df = get_feature_importance(best_model, FEATURES)

print("\nFeature Importance:")
display(importance_df)

plot_feature_importance(importance_df)

sample = pd.DataFrame([{
    "soil_moisture_%": 25,
    "temperature_C": 25,
    "rainfall_mm": 15,
    "humidity_%": 35,
    "sunlight_hours": 9,
    "NDVI_index": 0.42
}])

predict_sample(best_model, sample)

top_feature = importance_df.iloc[0]["Feature"]
print("\nMain factor influencing model decisions:", top_feature)

# AI-Powered Smart Irrigation Prediction System

### Applying Machine Learning to Smarter Water Management in Agriculture 

## Introduction

Water management has become one of the biggest challenges in modern agriculture. Many farms still rely on fixed irrigation schedules or manual observation, which can sometimes lead to overwatering or inefficient use of resources.

The goal of this project was to explore whether machine learning could help predict when crops actually require irrigation using environmental sensor data.

Using variables such as soil moisture, rainfall, humidity, sunlight exposure, temperature, and NDVI vegetation data, multiple machine learning models were trained and evaluated to determine which approach produced the most reliable irrigation predictions. 

## Dataset Overview

The dataset contains environmental and agricultural measurements collected from smart farming sensors.

Features used in this project include:
- Soil moisture percentage
- Temperature
- Rainfall
- Humidity
- Sunlight exposure
- NDVI vegetation index

These variables were used to help estimate whether irrigation would likely be needed under certain farming conditions.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

DATA_PATH = "/kaggle/input/datasets/atharvasoundankar/smart-farming-sensor-data-for-yield-prediction/Smart_Farming_Crop_Yield_2024.csv"

RANDOM_STATE = 42

FEATURES = [
    "soil_moisture_%",
    "temperature_C",
    "rainfall_mm",
    "humidity_%",
    "sunlight_hours",
    "NDVI_index"
]

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## Feature Engineering

To better represent irrigation demand, a custom feature called `water_score` was created.

The score increases when:
- soil moisture is low,
- temperature is high,
- humidity is low,
- rainfall is limited,
- and sunlight exposure is high.

This score was then used to generate a binary target variable called `needs_water`.

- `1` = irrigation recommended
- `0` = irrigation not necessary

In [ ]:
df["water_score"] = (
    (100 - df["soil_moisture_%"]) * 0.45 +
    df["temperature_C"] * 0.25 +
    (100 - df["humidity_%"]) * 0.15 +
    (100 - df["rainfall_mm"]) * 0.10 +
    df["sunlight_hours"] * 0.05
)

df["needs_water"] = (
    df["water_score"] > df["water_score"].median()
).astype(int)

display(df.head())

## Exploratory Data Analysis

Before training the machine learning models, exploratory analysis was performed to better understand the dataset structure and environmental relationships.

Visualizations were used to examine:
- feature distributions,
- class balance,
- and correlations between environmental variables.

This step helps identify patterns that may influence irrigation prediction performance.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nTarget balance:")
print(df["needs_water"].value_counts(normalize=True))

counts = df["needs_water"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Do Not Water", "Water"], counts.values)
plt.title("Target Class Balance")
plt.ylabel("Number of Samples")
plt.show()

for feature in FEATURES:
    plt.figure(figsize=(6, 4))
    plt.hist(df[feature], bins=30)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()

corr = df[FEATURES + ["needs_water"]].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Feature Correlation Matrix")
plt.show()

## Machine Learning Models

Three different machine learning models were trained and evaluated:

1. Logistic Regression
2. Random Forest Classifier
3. Gradient Boosting Classifier

These models were selected because they represent different machine learning approaches, including linear modeling and ensemble learning.

The dataset was divided into training and testing subsets using stratified sampling to preserve class balance.

In [ ]:
X = df[FEATURES]
y = df["needs_water"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=RANDOM_STATE
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nModels initialized:")
for name in models:
    print("-", name)

## Model Evaluation

Model performance was evaluated using several classification metrics:

- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix

F1 Score was emphasized because irrigation prediction requires balancing both false positives and false negatives.

This creates a more reliable evaluation than accuracy alone.

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds)
    })

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)
    print(classification_report(y_test, preds))

    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Do Not Water", "Water"]
    )

    disp.plot()
    plt.title(f"Confusion Matrix: {name}")
    plt.show()

results_df = pd.DataFrame(results).sort_values(
    by="F1 Score",
    ascending=False
)

display(results_df)

plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["F1 Score"])
plt.title("Model Comparison by F1 Score")
plt.ylabel("F1 Score")
plt.xticks(rotation=20)
plt.show()

## Feature Importance Analysis

Feature importance analysis was used to identify which environmental variables most strongly influenced irrigation predictions.

Understanding feature importance improves:
- model interpretability,
- agricultural insight,
- and trust in AI-assisted decision systems.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Best Performing Model:", best_model_name)

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring="f1"
)

print("5-Fold Cross-Validation F1 Scores:", cv_scores)
print("Average CV F1 Score:", round(cv_scores.mean(), 4))

if isinstance(best_model, Pipeline):
    final_model = best_model.named_steps["model"]
    importances = np.abs(final_model.coef_[0])
else:
    importances = best_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

display(importance_df)

plt.figure(figsize=(8, 5))
plt.barh(
    importance_df["Feature"],
    importance_df["Importance"]
)

plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()

## Sample Irrigation Prediction

A sample farm scenario was tested using custom environmental inputs.

The trained model predicts:
- whether irrigation is recommended,
- confidence level of the prediction,
- and the most influential environmental factor contributing to the decision.

In [ ]:
sample = pd.DataFrame([{
    "soil_moisture_%": 25,
    "temperature_C": 25,
    "rainfall_mm": 15,
    "humidity_%": 35,
    "sunlight_hours": 9,
    "NDVI_index": 0.42
}])

prediction = best_model.predict(sample)[0]
confidence = best_model.predict_proba(sample)[0][1]

display(sample)

if prediction == 1:
    print("AI Decision: Water the plant")
else:
    print("AI Decision: Do not water")

print(
    "Confidence that watering is needed:",
    round(confidence * 100, 2),
    "%"
)

top_feature = importance_df.iloc[0]["Feature"]

print(
    "Main factor influencing model decisions:",
    top_feature
)

## Results

The machine learning models successfully identified irrigation patterns using environmental sensor data.

Among the evaluated models, ensemble-based methods such as Random Forest and Gradient Boosting demonstrated the strongest performance due to their ability to capture nonlinear agricultural relationships.

The results demonstrate the potential of machine learning systems for precision agriculture and intelligent water management.

## Future Improvements

Several future improvements could further enhance the system:

- Real-time IoT sensor integration
- Weather forecasting support
- Satellite imagery analysis
- Drone-based crop monitoring
- Deep learning models
- Automated irrigation systems
- Mobile dashboard deployment
- Live environmental data streaming

Future versions could evolve into a larger AI-assisted smart farming platform.

## Conclusion

This project explored the use of machine learning for intelligent irrigation prediction using environmental sensor data.

By combining predictive modeling with agricultural measurements, machine learning systems may help improve:
- water efficiency,
- crop management,
- and agricultural sustainability.

The project highlights the growing role artificial intelligence can play in precision agriculture and environmental resource optimization.